# 03 — CV à l'échelle du leaderboard + toutes les features

Deux corrections majeures par rapport aux notebooks 01/02 :
1. **AP calculée sur le test COMPLET** (op_03 + non-op_03 mis à 0), comme le LB.
   Les notebooks précédents calculaient l'AP sur op_03 seulement -> chiffres non comparables au LB.
2. **Métrique de référence = folds récents** (`recent_mean`, `last`), pas la moyenne globale.

Features : row-level + solde (fingerprints PaySim) + comportemental + dynamique récente,
toutes apprises sur le passé strict (anti-fuite).

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from src import config as C
from src.validation import time_folds, evaluate_ap
from src.utils import op03_mask, make_submission, seed_everything
from src.calibration import fit_isotonic, apply_isotonic
from src.features.temporal import balance_features, recency_features
from src.features.behavioral import behavioral_features
seed_everything(42)
DATA = ROOT / "data"

In [ ]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
op03 = op03_mask(train).to_numpy()
y_all = train[C.TARGET].to_numpy()
print("train:", train.shape, "| op_03:", int(op03.sum()), "| prévalence globale:", round(y_all.mean(), 4))

In [ ]:
EPS = 1e-6

def row_features(df):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]
    f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    return pd.concat([f, balance_features(df)], axis=1)

def add_freq(X, src_df, ref_df):
    X = X.copy()
    for col in [C.ORIGIN_ACCT, C.DEST_ACCT]:
        freq = ref_df[col].value_counts(normalize=True)
        X[f"freq_{col}"] = src_df[col].map(freq).fillna(0).values
    return X

def build_features(df, ref):
    """Toutes les features pour `df`, agrégations apprises sur `ref` (= passé op_03)."""
    X = row_features(df).reset_index(drop=True)
    X = add_freq(X, df.reset_index(drop=True), ref)
    beh = behavioral_features(df, ref).reset_index(drop=True)
    rec = recency_features(df, ref).reset_index(drop=True)
    return pd.concat([X, beh, rec], axis=1)

def make_model():
    try:
        from catboost import CatBoostClassifier
        return CatBoostClassifier(loss_function="Logloss", eval_metric="PRAUC", depth=6,
                                  learning_rate=0.05, iterations=600, random_seed=42, verbose=False)
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingClassifier
        return HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, random_state=42)

## CV temporelle, AP op_03 *et* AP test-complet par fold
Folds définis sur le train COMPLET. Le modèle n'apprend que sur op_03 ; les non-op_03
de validation reçoivent une proba 0. On mesure les deux AP.

In [ ]:
folds_full = list(time_folds(train[C.PERIOD]))
oof_full = np.zeros(len(train))
ap_op03, ap_lb = [], []
last_model, last_cols = None, None

for k, (tr_idx, va_idx) in enumerate(folds_full):
    tr_op = tr_idx[op03[tr_idx]]
    va_op = va_idx[op03[va_idx]]
    ref = train.iloc[tr_op]
    Xtr = build_features(train.iloc[tr_op], ref)
    Xva = build_features(train.iloc[va_op], ref)
    m = make_model(); m.fit(Xtr, y_all[tr_op])
    p = m.predict_proba(Xva)[:, 1]
    oof_full[va_op] = p
    a_op = evaluate_ap(y_all[va_op], p)
    a_lb = evaluate_ap(y_all[va_idx], oof_full[va_idx])  # inclut les non-op_03 (proba 0)
    ap_op03.append(a_op); ap_lb.append(a_lb)
    last_model, last_cols = m, Xtr.columns
    print(f"fold {k}: AP op_03 = {a_op:.4f} | AP test-complet (LB) = {a_lb:.4f}")

print("\n--- Synthèse (ce qui compte = récent) ---")
print(f"AP op_03   : global {np.mean(ap_op03):.4f} | recent(2) {np.mean(ap_op03[-2:]):.4f} | last {ap_op03[-1]:.4f}")
print(f"AP LB-scale: global {np.mean(ap_lb):.4f} | recent(2) {np.mean(ap_lb[-2:]):.4f} | last {ap_lb[-1]:.4f}")
print("\n=> Le 'last' de la ligne LB-scale est notre meilleure estimation du score public.")

## Importance des features (dernier fold)

In [ ]:
if hasattr(last_model, "get_feature_importance"):
    imp = last_model.get_feature_importance()
else:
    imp = getattr(last_model, "feature_importances_", np.zeros(len(last_cols)))
print(pd.Series(imp, index=last_cols).sort_values(ascending=False).round(2).head(15))

## Modèle final + soumission (toutes features)
Entraîné sur tout le train op_03, calibré isotonic, proba 0 hors op_03.

In [ ]:
ref_full = train.iloc[np.where(op03)[0]]
X_full = build_features(ref_full, ref_full)
y_full = y_all[op03]
final = make_model(); final.fit(X_full, y_full)
iso = fit_isotonic(oof_full[op03], y_full)

te_op = op03_mask(test).to_numpy()
X_te = build_features(test.iloc[np.where(te_op)[0]], ref_full)
proba_te = apply_isotonic(iso, final.predict_proba(X_te)[:, 1])

full = np.zeros(len(test))
full[te_op] = proba_te
path = make_submission(test[C.ID], full, "03_full_features")
sub = pd.read_csv(path)
assert list(sub.columns) == ["id", "target"] and len(sub) == len(test)
assert set(sub["id"]) == set(sample["id"]) and sub["target"].between(0, 1).all()
print("soumission écrite :", path, "| proba>0 :", int((sub['target'] > 0).sum()))